# Husky + TraceWhisperer Explorer

This notebook uses TraceWhisperer in a similar way to our [TraceWhisperer.ipynb notebook](https://github.com/newaetech/DesignStartTrace/blob/master/jupyter/TraceWhisperer.ipynb), but in an interactive manner that allows you to better "see" Arm trace in action.

While our other trace notebook cover the many knobs and dials of TraceWhisperer (different targets and platforms, raw trace capture vs pattern match captures, SWO vs parallel interface, PC sampling, DWT events...), here we focus on a single scenario: 
- **SWO interface.** While the parallel trace interface is more powerful (due to its much higher bandwidth), not many of our targets have it, so here we showcase the much more common SWO interface.
- **Pattern match mode.** Similarly, while the raw capture mode can be much more powerful, Arm's raw trace format is not nice to work with at all! In our other notebooks we show how you can use a third-party software ([Orbuculum](https://github.com/orbcode/orbuculum)) to handle the parsing of the raw trace data, but this introduces complexities and dependencies; you also can't trigger a capture or a glitch event from trace data if you need software in the middle to interpret it. Here we show that for many use cases, parsing is not actually necessary.
- **`DWT_COMP` match events**. Arm trace can generate trace data for a gazillion things. Here we look at only one of these things, but a particularly useful one for side-channel work: when does the PC reach the values programmed in the `DWT_COMP0` and/or `DWT_COMP1` registers.
- **Single target, single platform**. The STM32 is our most common target which supports the trace features required for this demo. And while [TraceWhisperer can run on other platforms](https://github.com/newaetech/tracewhisperer) (CW305 and PhyWhisperer), you're here because you have a Husky.

While this is far from the only thing you can do with TraceWhisperer, it is the original motivation for developing it: it allows you to find arbitrary instructions in your power trace.

## Requirements:
- Husky (or Husky Plus)
- **STM32F3** target
- pre-compiled simpleserial-trace FW
- jumper cables to connect target TMS/TCK/TDO to Husky USERIO D0/D1/D2

Unfortunately, this notebook **cannot be used with the SAM4S target** (the SAM4S is unable to output the type of trace data that we need in this demo, because it has no ETM).

Other TraceWhisperer notebooks in our [DesignStartTrace repository](https://github.com/newaetech/DesignStartTrace) do have broader target support, so look there if you don't have an STM32 target handy.

# Setup

In [ ]:
import chipwhisperer as cw

In [ ]:
PLATFORM = 'CW308_STM32F3' # no other options allowed!
SCOPETYPE = 'OPENADC'

%run "../../Setup_Scripts/Setup_Generic.ipynb"
scope.trace.target = target
scope.adc.samples = 31000

scope.trace.enabled = True
scope.gain.db = 12

In [ ]:
reset_target(scope)

While any firmware could be used in theory, in order to have everything "just work" we use a specific pre-compiled binary:

In [ ]:
if scope.trace.get_fw_buildtime() != 'ChipWhisperer simpleserial-trace, compiled Mar 14 2022, 21:06:34':
    print('Programming target firmware...')
    fw_path = '../../../firmware/mcu/simpleserial-trace/simpleserial-trace-CW308_STM32F3.hex'
    prog = cw.programmers.STM32FProgrammer
    cw.program_target(scope, prog, fw_path)

## Switch to SWD mode.

Arm processors which support JTAG and SWD come out of reset in JTAG mode. In order to get trace data out of the SWO pin, we need to switch it over to SWD mode.

The `jtag_to_swd()` call below runs a special sequence on the TMS and TCK pins to do this switchover.

**This requires connecting the target's TMS/TCK/TDO pins to Husky's D0/D1/D2 pins.**

In [ ]:
scope.trace.clock.fe_clock_src = 'target_clock'
assert scope.trace.clock.fe_clock_alive, "Hmm, the clock you chose doesn't seem to be active."
scope.trace.trace_mode = 'SWO'
scope.trace.jtag_to_swd() # switch target into SWO mode

acpr = 0
trigger_freq_mul = 8
scope.trace.clock.swo_clock_freq = scope.clock.clkgen_freq * trigger_freq_mul
scope.trace.target_registers.TPI_ACPR = acpr
scope.trace.swo_div = trigger_freq_mul * (acpr + 1)
assert scope.trace.clock.swo_clock_locked, "Trigger/UART clock not locked"
assert scope.userio.pins[2].status, "SWO line not high"

#### Check that the target is alive.
If `get_fw_buildtime()` produces no output, the target may have become unresponsive after the above changes; it may simply require a reset.

In [ ]:
if scope.trace.get_fw_buildtime() != 'ChipWhisperer simpleserial-trace, compiled Mar 14 2022, 21:06:34':
    reset_target(scope)
    time.sleep(0.1)
    assert scope.trace.get_fw_buildtime() == 'ChipWhisperer simpleserial-trace, compiled Mar 14 2022, 21:06:34'

### Disable sync frames for SWO:

By default, periodic sync frames are emitted every 16 million clock cycles. If you're bringing up an SWO target for the first time, this is helpful to confirm that the link is "alive".
However these sync frames will delay the trace events that we care about if they occur during our trace capture, so it's best to disable them.

In [ ]:
scope.trace.target_registers.DWT_CTRL = 0x40000021

### Trigger trace capture from target FW:

While it's also possible (and very useful!) to trigger from the trace events themselves, here we simply use the static firmware-generated trigger on the TIO4 pin.

This allows us to better see how trace events occur and move relative to a single fixed reference.

In [ ]:
scope.trace.capture.trigger_source = 'firmware trigger'

### What to capture:

[Other TraceWhisperer notebooks](https://github.com/newaetech/DesignStartTrace/tree/master/jupyter) show how to use the "raw" trace capture mode; here we only use the non-raw mode capture mode, where TraceWhisperer captures event timestamps whenever it sees trace activity that matching a given pattern.

This saves us from the difficult task of fully decoding the complex trace data format, and it works very well!

All trace event packets that we will be generating here (`DWT_COMPx` match events) begin with `[3, 8, 32]`, so that's what we set our pattern match to.

In [ ]:
scope.trace.capture.raw = False
scope.trace.set_pattern_match(0, [3, 8, 32])
# enable matching rule:
scope.trace.capture.rules_enabled = [0]

### How long to capture for:
We set TraceWhisperer to capture trace events only when the target's trigger line is high.

In [ ]:
scope.trace.capture.mode = 'while_trig'

### Set PC addresses to match on:
Let's use the start of the `SubBytes()` and `AddRoundKey()` functions.

The `set_isync_matches()` function sets the target's `DWT.COMP0`, `DWT.COMP1`, and `ETM.TEEVR` registers.

(If you recompiled the firmware, you may need to adjust these addresses accordingly - this is why we suggest starting with our pre-compiled firmware.)

In [ ]:
scope.trace.set_isync_matches(addr0=0x080018c4, addr1=0x0800188c, match='both')

In this demo we don't want periodic PC sampling events; we want all the bandwidth of the SWO link available for our `DWT_COMPx` match events:

In [ ]:
scope.trace.set_periodic_pc_sampling(enable=0)

## Capture a first power and debug trace:

We're finally all set for our first trace:

In [ ]:
# arm trace sniffer:
scope.trace.arm_trace()

powertrace = cw.capture_trace(scope, target, bytearray(16), bytearray(16))
assert powertrace is not None, 'capture failed; try resetting the target?'

Read the raw trace data:

In [ ]:
assert not scope.trace.fifo_empty(), 'TraceWhisperer capture failed! Try resetting the target?'
raw = scope.trace.read_capture_data()
times = scope.trace.get_rule_match_times(raw, rawtimes=False, verbose=True)

Plot:

In [ ]:
from bokeh.plotting import figure, show
from bokeh.io import output_notebook
from bokeh.models import Span

output_notebook()
p = figure(width=1200)

xrange = list(range(scope.adc.samples))
p.line(xrange, powertrace.wave, line_color="red")

multiplier = scope.clock.adc_mul
vlines = []
for t in times:
    vlines.append(Span(location=t[0]*multiplier, dimension='height', line_color='black', line_width=2))
p.renderers.extend(vlines)

show(p)

The power trace is the AES operation (same as our usual `simpleserial-aes` firmware).

The vertical lines indicate the trace match events; remember that we set the target to emit a trace event whenever the PC matches `0x080018c4` or `0x0800188c`, which are the start addresses of the `SubBytes()` and `AddRoundKey()` functions.

While the AES rounds of `simpleserial-aes` are visible in a normal capture without TraceWhisperer, here we can clearly see where different parts of the AES algorithm are being executed.

# Interactive TraceWhisperer Exploration

If everything above worked, we're finally ready to explore the capabilities of Arm Trace and our TraceWhisperer in more detail with the help of `TraceWhispererExplorer`.

In this demo, as we did above in our first capture, we're using the Arm trace hardware to learn when certain PC addresses are executed.

This has a lot more meaning if you have the firmware disassembly at hand, so let's grab that:

In [ ]:
%%capture dumpall
%%bash
arm-none-eabi-objdump -d ../../../firmware/mcu/simpleserial-trace/simpleserial-trace-CW308_STM32F3.elf

If you look at the disassembly (`print(dumpall)`) you'll find a lot of functions that we don't care about because they never get executed when our target is encrypting (for example: `get_key()` only runs when the key is sent to the target, well ahead of the encryption).

So let's define a list of functions that we do potentially care about:

In [ ]:
useful_functions = ['KeyExpansion',
                    'AddRoundKey',
                    'SubBytes',
                    'ShiftRows',
                    'xtime',
                    'Cipher',
                    'BlockCopy',
                    'AES128_ECB_indp_crypto']

Now we parse the disassembly to build up an array of these functions and their disassembly; `TraceWhispererExplorer` will use this data to help you select PC addresses:

In [ ]:
import re
head_regex = re.compile(r'(\w{8})\s<(\w+)>:$')
diss_regex = re.compile(r'\s(\w+):')
matches = 0
mismatches = 0
funcs = []

dumplines = []
for l in dumpall.stdout.split('\n'):
    dumplines.append(l)

for i in range(len(dumplines)):
    line = dumplines[i]
    match = head_regex.search(line)
    if match:
        start = match.group(1)
        name = match.group(2)
        if name not in useful_functions:
            continue
        matches += 1
        j = 1
        diss = ''
        while (True):
            nextline=dumplines[i+j]
            match = diss_regex.search(nextline)
            if match:
                diss += nextline + '\n'
                addr = match.group(1)
                j += 1
            else:
                i += j
                break
        funcs.append([int(start, 16), int(addr, 16), diss, name])

assert matches == len(useful_functions)

In [ ]:
from chipwhisperer.common.utils.tracewhisperer_explorer import TraceWhispererExplorer

We are finally ready to play!

When you create a `TraceWhispererExplorer` object below, you get a few interactive fields to play with:
- **DWT_COMP0 value**: the first PC address match register
- **DWT_COMP1 value**: the second PC address match register
- **comparators**: select 0 to enable only DWT_COMP0, 1 to enable only DWT_COMP1, etc...
- **TPI.ACPR value**: this controls the rate on the SWO line; 0 is fastest.
- **dissassemble**: a drop-down menu to select which function's disassembly you want to see
- **include trace activity "noise"**: trace activity on the SWO line can cause significant spikes on the power trace. When this option is deselected, we run two captures: one with trace enabled, to gather the trace data, and a second capture with trace disabled, to get a less noisy power trace. The trace data from the noisy capture is still relevant to the noiseless no-trace capture because we are making the target do the exact same thing for each capture.
- **capture SWD activity**: when this is selected, an additional capture is run where `scope.LA` is used to capture the SWO line and show it alongside the power trace. This allows you to better understand the bandwidth limitations of trace. This requires an additional capture because `scope.LA` and `scope.trace` are mutually exclusive: they can't be used simultaneously. Because `scope.LA` has limited storage capacity, only SWD activity for the first part of the power trace is collected.
- **run the capture**: when this is selected, any change to the other widgets will trigger a capture (this is an ipywidgets quirk)

When you run a capture, the plot background color changes as follows:
1. When the capture is initiated, it changes momentarily to **yellow**.
2. If the capture is successful, it changes momentarily to **green**.
3. If any part of the capture fails (e.g. no trace data), it changes to **red** (and remains red until the next capture). Any data that *was* captured successfully will still be plotted (e.g. the power trace, when the debug trace capture fails).
4. After a successful capture, once the plot is updated, it returns to **white**.

Finally, a reminder that the default `DWT_COMPx` values are:
- `0x080018c4`: start of `SubBytes()`
- `0x0800188c`: start of `AddRoundKey()`

In [ ]:
te = TraceWhispererExplorer(scope, target, funcs, width=1600, height=400)

Try different things!

In particular this demo is useful to understand the bandwidth limitations of trace.

For example, with the default DWT_COMPx values of `0x080018c4` and `0x0800188c`, watch the number of trace events go down as `TPI.ACPR` is increased.

Similarly, if you pick two addresses that are "too close" together, you'll probably only get events for one of them.

On certain instructions, the Arm trace hardware appears not issue any events at all, even if logic dictates that instruction at the specified address *is* getting executed. This is not a TraceWhisperer bug; it is purely an Arm "feature". Select "capture SWD activity" to confirm that the SWO line is idle.

Finally, if you find the target has become unresponsive, it may simply need a reset:

In [ ]:
reset_target(scope)